<!-- nav -->
[← 5 · One framework, three languages](05_One_Framework_Three_Languages.ipynb) · [Home](../../README.md) · [7 · From one language to another →](07_From_One_Language_To_Another.ipynb)

# 6 · Trees on the wire

**The problem.** The analyzer is a Python program; the editor plugin that shows its findings is TypeScript. Both need
the same tree. Parsing twice risks two parsers disagreeing, and an ad hoc JSON format is one more thing to maintain.

Trees are mbse-schemas data: every kind has a meta-schema, so a tree saves and loads as JSON or YAML like any object
graph, and the Python and TypeScript implementations write it byte for byte the same.

In [1]:
import { createHash } from "node:crypto";

import { same } from "@mbse/programs/Framework/Syntax";
import { Python314, Syntax as P } from "@mbse/programs/Python";
import { JSON as SchemaJSON } from "@mbse/schemas/Framework";

const module = Python314.parse("def area(w, h):\n    return w * h\n");

## Step 1: a snapshot

`SchemaJSON.ToJSON(P.LANGUAGE.Builders)` writes a tree reachable from its root. Each syntax node is an object tagged
with its `kind`, holding its attributes, and its children are entries naming the property they fill:

In [2]:
const text = SchemaJSON.ToJSON(P.LANGUAGE.Builders).Reachable(P.Module.Schema, module, { indent: 1 });
console.log(text.slice(0, 900), "...");

{
 "root": "s0",
 "objects": {
  "s0": {
   "kind": "Module",
   "children": [
    {
     "child": {
      "$ref": "s1",
      "$schema": "Programs.Python.FunctionDef"
     },
     "property": "body",
     "index": 0
    }
   ]
  },
  "s1": {
   "kind": "FunctionDef",
   "children": [
    {
     "child": {
      "$ref": "s2",
      "$schema": "Programs.Python.Identifier"
     },
     "property": "name"
    },
    {
     "child": {
      "$ref": "s3",
      "$schema": "Programs.Python.Arguments"
     },
     "property": "args"
    },
    {
     "child": {
      "$ref": "s4",
      "$schema": "Programs.Python.Return"
     },
     "property": "body",
     "index": 0
    }
   ]
  },
  "s2": {
   "kind": "Identifier",
   "spelling": "area"
  },
  "s3": {
   "kind": "Arguments",
   "children": [
    {
     "child": {
      "$ref": "s5",
      "$schema": "Programs.Python.Arg"
     },
     "prop ...


## Step 2: reading it back

`SchemaJSON.FromJSON` builds the tree again. `same` compares trees by structure, since syntax nodes are compared by
identity:

In [3]:
const loaded = SchemaJSON.FromJSON(P.LANGUAGE.Builders).Reachable(P.Module.Schema, text) as P.Module;
console.log(loaded === module, same(loaded, module));
console.log(Python314.print(loaded));

false true


def area(w, h):
    return w * h



## Step 3: the same bytes in both languages

The Python implementation parses the same source to the same tree, and writes the same snapshot. Their fingerprints
show it: the Python tutorial prints this same digest.

In [4]:
const bigger = Python314.parse(`import math


class Circle:
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2


print(f"{Circle(2).area():.2f}")
`);
const snapshot = SchemaJSON.ToJSON(P.LANGUAGE.Builders).Reachable(P.Module.Schema, bigger);
console.log(createHash("sha256").update(snapshot).digest("hex"));

a295482fb0b47e51de6178d758c923e16dce53c8d5999e2a76c263659c3577ae


## Step 4: the grammar is data too

A language describes itself: every kind, its category, its properties and where it exists. A tool in any language can
read it, to generate code or documentation:

In [5]:
const grammar = P.LANGUAGE.grammar() as any;
console.log(grammar.kinds.find((kind: any) => kind.kind === "BinOp"));

{
  kind: "BinOp",
  category: "Expression",
  since: null,
  extension: false,
  features: { op: { "@": { Python: 305 } } },
  properties: [
    {
      name: "left",
      type: [ "Expression" ],
      optional: false,
      many: false
    },
    {
      name: "op",
      type: "str",
      optional: false,
      choices: [
        "+",  "-",  "*", "@",
        "/",  "//", "%", "**",
        "<<", ">>", "|", "^",
        "&"
      ]
    },
    {
      name: "right",
      type: [ "Expression" ],
      optional: false,
      many: false
    }
  ]
}


## Why it works this way

- **One data model.** Trees are mbse-schemas objects, so they get its JSON and YAML, validation and comparison for
  free, and so does every tool built on mbse-schemas.
- **Byte-identical, by test.** Both implementations' snapshots of the shared corpus are compared byte for byte (the
  conformance suite), so a tree written by one reads in the other exactly.
- **Children as entries.** Each child records the property it fills and its index, so a snapshot keeps the tree's
  order and structure without a custom format.

<!-- nav -->
[← 5 · One framework, three languages](05_One_Framework_Three_Languages.ipynb) · [Home](../../README.md) · [7 · From one language to another →](07_From_One_Language_To_Another.ipynb)